In [1]:
from utils_00 import *
# exec(open('c00_utils.py').read())

pn.extension()  # 启用 Jupyter 支持
pd.set_option('expand_frame_repr', False)
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', 100)

In [2]:
native_exchange_data_dir = os.path.join(base_dir, "Native_Exchange_Data")
temporary_experimental_data_dir = os.path.join(base_dir, "Temporary_Experimental_Data")
pro = ts.pro_api(secret_dict['tushare']['api_key'])

def get_exchange_variety_map():
    # 交易所日历路径 NED_dimension_exchange_calendar_path
    NED_dimension_exchange_calendar_path = os.path.join(native_exchange_data_dir, 'dimension_exchange_calendar.csv')
    dimension_exchange_calendar = pd.read_csv(NED_dimension_exchange_calendar_path, dtype = {'date': str})

    # 获取 交易所-品种 表单
    exchange_list = [col for col in dimension_exchange_calendar.columns if col != 'date'] # 交易所列表
    map_list = []
    for exchange in exchange_list:

        exchange_variety_map = pd.DataFrame()
        for attempt in range(3):
            try:
                exchange_variety_map = pro.fut_basic(exchange = exchange, fut_type = "1", fields = ["fut_code"])
                break
            except Exception as e:
                print(f"第 {attempt + 1} 次获取 {exchange} 品种信息失败: {e}")

        exchange_variety_map = exchange_variety_map.drop_duplicates()
        exchange_variety_map["exchange"] = exchange
        map_list.append(exchange_variety_map)

    exchange_variety_map = pd.concat(map_list, ignore_index = True)
    exchange_variety_map = exchange_variety_map.groupby('exchange')['fut_code'].apply(list)
    exchange_variety_map = exchange_variety_map.to_dict()
    return exchange_variety_map

if __name__ == '__main__':
    exchange_variety_map = get_exchange_variety_map()
    exchange_variety_map.pop("CZCE")
    exchange_variety_map.pop("DCE")
    print(exchange_variety_map)

{'CFFEX': ['T', 'TF', 'IF', 'IM', 'IC', 'IH', 'TS', 'TL'], 'GFEX': ['LC', 'SI', 'PS', 'PD', 'PT'], 'INE': ['NR', 'SC', 'LU', 'BC', 'SCTAS', 'EC'], 'SHFE': ['AO', 'NI', 'AL', 'HC', 'BU', 'RU', 'FU', 'CU', 'AG', 'ZN', 'AU', 'PB', 'SS', 'RB', 'SN', 'WR', 'SP', 'BR', 'AD', 'OP']}


In [3]:
exchange = None
variety = None

if __name__ == '__main__':
    # 交易所选择组件
    exchange_radio = widgets.RadioButtons(
        options = list(exchange_variety_map.keys()),
        description = '交易所选择 exchange:',
        layout = {'width': '400px'}
    )

    # 品种选择组件（初始禁用）
    species_toggle = widgets.ToggleButtons(
        options = [],
        description = '品种选择 variety:',
        disabled = True,
        button_style = '',
        style = {'button_width': 'auto'},
        layout = widgets.Layout(width = '600px'),
    )

    # 输出提示区域
    out = widgets.Textarea(
        value = '请先选择交易所，然后选择品种',
        layout = {'width': '400px', 'height': '50px'},
        disabled = True
    )

    display(widgets.VBox([exchange_radio, species_toggle, out]))

    def on_exchange_change(change):
        """当交易所改变时，更新可选品种"""
        global exchange
        if change['name'] == 'value':
            exchange = change['new']
            varieties = exchange_variety_map.get(exchange, [])
            species_toggle.options = sorted(varieties)
            species_toggle.disabled = len(varieties) == 0
            if varieties:
                species_toggle.value = varieties[0]  # 默认选第一个
                out.value = f'已选择交易所：{exchange}，请选择品种'
            else:
                out.value = f'交易所 {exchange} 无可用品种'

    def on_species_change(change):
        """当品种改变时，记录到全局变量"""
        global variety
        if change['name'] == 'value' and species_toggle.options:
            variety = change['new']
            out.value = f'已选择交易所 exchange: {exchange}\n已选择品种 variety: {variety}'

    # 绑定事件 ["SCTAS", "SI", "FU", "AU", "SS", "WR"]
    exchange_radio.observe(on_exchange_change, names = 'value')
    species_toggle.observe(on_species_change, names = 'value')

In [4]:
dataset_range_boundary = ""  # 选择的时间范围边界字符串

if __name__ == '__main__':

    # 临时实验数据品种文件夹路径 TED_variety_path
    TED_variety_path = os.path.join(str(temporary_experimental_data_dir), str(variety))
    dataset_range = os.listdir(TED_variety_path)
    dataset_range = sorted(dataset_range)

    dataset_range_buttons = []  # 创建数据集范围边界按钮容器
    for range_file in dataset_range:  # 为每个范围文件创建按钮
        btn = widgets.Button(
            description = range_file,
            layout = {'width': '530px', 'height': '25px'},  # 宽度较大以显示完整文件名
            button_style = ''
        )
        dataset_range_buttons.append(btn)

    dataset_range_button_column = widgets.VBox(dataset_range_buttons) # Horizontal Box & Vertical Box

    # 输出显示区域
    range_display_output = widgets.Textarea(
        value = 'dataset_range_boundary: 未选择',
        layout = {'width': '530px', 'height': '30px'},
        disabled = True
    )

    # 显示所有组件
    display(widgets.VBox([
        widgets.Label('数据集范围边界 dataset_range_boundary:'),
        dataset_range_button_column,
        range_display_output
    ]))

    def update_range_display():
        """更新显示区域"""
        global dataset_range_boundary
        range_display_output.value = f'dataset_range_boundary: {dataset_range_boundary}'

    def set_dataset_range_boundary(range_file):
        """设置数据集范围边界"""
        global dataset_range_boundary
        dataset_range_boundary = range_file
        update_range_display()

    # 事件处理函数
    def on_dataset_range_button_click(btn):
        """数据集范围按钮 点击事件"""
        set_dataset_range_boundary(btn.description)

    # 绑定事件
    for btn in dataset_range_buttons:
        btn.on_click(on_dataset_range_button_click)

    # 初始化
    update_range_display()

In [5]:
tick_granularity = 0  # 采样颗粒度 全局变量
resample_axis = None  # 采样轴 全局变量

if __name__ == '__main__':

    # 创建 tick_granularity 输入框组件
    tick_granularity_input = widgets.IntText(
        value = 0,
        description = 'tick_granularity:',
        layout = {'width': '250px', 'height': '25px'},
        style = {'description_width': '90px'},
        continuous_update = False
    )

    # 创建 tick_granularity 按钮容器
    granularity_buttons = []
    for i in [i for i in range(0, 181, 15)]:
        btn = widgets.Button(
            description = str(i),
            layout = {'width': '45px', 'height': '25px'}
        )
        btn.value = i
        granularity_buttons.append(btn)
    granularity_button_row = widgets.HBox(granularity_buttons)

    # tick_granularity 输出显示区域
    granularity_out = widgets.Textarea(
        value = '当前 tick_granularity: 0',
        layout = {'width': '400px', 'height': '30px'},
        disabled = True
    )

    # 创建 resample_axis 按钮
    axis_buttons = []
    axis_options = ['time_axis_data', 'volume_axis_data', 'money_axis_data']  # 重采样刻度选项
    for option in axis_options:
        btn = widgets.Button(
            description = option,
            layout = {'width': '135px', 'height': '25px'}
        )
        btn.value = option
        axis_buttons.append(btn)
    axis_button_row = widgets.HBox(axis_buttons)

    # resample_axis 显示区域
    axis_out = widgets.Textarea(
        value = '当前 resample_axis: None',
        layout = {'width': '400px', 'height': '30px'},
        disabled = True
    )

    # 显示所有组件
    display(widgets.VBox([tick_granularity_input, granularity_button_row, granularity_out, axis_button_row, axis_out]))

    def update_display(value):
        """更新显示"""
        global tick_granularity
        tick_granularity = value
        granularity_out.value = f'当前 tick_granularity: {tick_granularity}'
        tick_granularity_input.value = value

    def update_axis_display(value):
        """更新 resample_axis 显示"""
        global resample_axis
        resample_axis = value
        axis_out.value = f'当前 resample_axis: {resample_axis}'

    def on_input_change(change):
        """当输入框值改变时"""
        if change['name'] == 'value':
            new_value = change['new']
            if 0 <= new_value <= 600:
                update_display(new_value)
            else:
                tick_granularity_input.value = tick_granularity  # 恢复原值

    def on_button_click(btn):
        """当按钮点击时"""
        new_value = btn.value
        update_display(new_value)

    def on_axis_button_click(btn):
        """当 resample_axis 按钮点击时"""
        new_value = btn.value
        update_axis_display(new_value)

    # 绑定输入框事件
    tick_granularity_input.observe(on_input_change, names ='value')

    # 绑定所有按钮事件
    for btn in granularity_buttons:
        btn.on_click(on_button_click)

    # 绑定 resample_axis 按钮事件
    for btn in axis_buttons:
        btn.on_click(on_axis_button_click)

In [6]:
lookback_months = 0  # 回溯月份数 全局变量
split_index = None  # 切割点 index 全局变量
split_time_str = None  # 切割时间字符串 全局变量

if __name__ == '__main__':

    TED_variety_path = os.path.join(str(temporary_experimental_data_dir), str(variety))  # 临时实验数据品种文件夹路径 TED_variety_path
    dataset_range_boundary_path = os.path.join(TED_variety_path, dataset_range_boundary)  # 时间范围文件夹路径 dataset_range_boundary_path
    tick_granularity_resample_path = os.path.join(dataset_range_boundary_path, f'tick_granularity_{tick_granularity}')  # 采样刻度文件路径 tick_granularity_resample_path
    resample_axis_data_path = os.path.join(tick_granularity_resample_path, resample_axis)  # 驱动轴数据文件路径 resample_axis_data_path

    tick_type_dict = {
        'time_axis_data': 'Nrow',
        'volume_axis_data': 'volume',
        'money_axis_data': 'money'
    }
    if resample_axis in tick_type_dict.keys():
        tick_type = tick_type_dict[resample_axis]
    else: raise ValueError

    # 主力片段拼接重采样时间索引路径 main_continuous_resampled_date_index_path
    main_continuous_resampled_date_index_path = os.path.join(resample_axis_data_path, rf"main_continuous_{tick_type}_date_index.csv")
    main_continuous_resampled_date_index = pd.read_csv(main_continuous_resampled_date_index_path, dtype = {'date': str})

    # 创建 lookback_months 输入框组件
    lookback_months_input = widgets.IntText(
        value = 0,
        description = 'lookback_months:',
        layout = {'width': '250px', 'height': '25px'},
        style = {'description_width': '100px'},
        continuous_update = False
    )

    # 创建 lookback_months 按钮容器（1~24）
    months_buttons = []
    for i in range(1, 25):
        btn = widgets.Button(
            description = str(i),
            layout = {'width': '45px', 'height': '25px'}
        )
        btn.value = i
        months_buttons.append(btn)
    first_row = widgets.HBox(months_buttons[:12])   # 前12个按钮
    second_row = widgets.HBox(months_buttons[12:])  # 后12个按钮
    months_button_row = widgets.VBox([first_row, second_row])

    # 输出显示区域：包含回溯月份、切割index、切割时间
    split_info_out = widgets.Textarea(
        value = '当前 lookback_months: \nsplit_index: None \nsplit_time_str: None',
        layout = {'width': '600px', 'height': '60px'},
        disabled = True
    )

    # 显示所有组件
    display(widgets.VBox([lookback_months_input, months_button_row, split_info_out]))

    def update_split_info():
        """根据 lookback_months 和 main_continuous_resampled_date_index 计算并更新 split_index 与 split_time_str"""
        global lookback_months, split_index, split_time_str

        if lookback_months <= 0 or main_continuous_resampled_date_index.empty:
            split_index = None
            split_time_str = None
        else:
            # 获取最后一行的时间字符串并转换为 datetime
            last_time_str = main_continuous_resampled_date_index['date'].iloc[-1]
            last_dt = pd.to_datetime(last_time_str)

            # 回溯指定月份数（注意：pandas 的 DateOffset 支持月份偏移）
            target_dt = last_dt - pd.DateOffset(months=lookback_months)

            # 找到最接近但不大于 target_dt 的行的 index
            date_series = pd.to_datetime(main_continuous_resampled_date_index['date'])
            valid_mask = date_series <= target_dt
            if valid_mask.any():
                split_index = valid_mask[::-1].idxmax()  # 最后一个满足条件的 index
                split_time_str = main_continuous_resampled_date_index['date'].iloc[split_index]
            else:
                split_index = 0
                split_time_str = main_continuous_resampled_date_index['date'].iloc[0]

        # 更新显示
        split_info_out.value = f'当前 lookback_months: {lookback_months} \nsplit_index: {split_index} \nsplit_time_str: {split_time_str}'

    def on_months_input_change(change):
        """当输入框值改变时"""
        if change['name'] == 'value':
            new_value = change['new']
            if 0 <= new_value <= 24:
                global lookback_months
                lookback_months = new_value
                lookback_months_input.value = new_value
                update_split_info()
            else:
                lookback_months_input.value = lookback_months  # 恢复原值

    def on_months_button_click(btn):
        """当按钮点击时"""
        global lookback_months
        lookback_months = btn.value
        lookback_months_input.value = lookback_months
        update_split_info()

    # 绑定输入框事件
    lookback_months_input.observe(on_months_input_change, names='value')

    # 绑定所有按钮事件
    for btn in months_buttons:
        btn.on_click(on_months_button_click)

    # 初始化显示
    update_split_info()

In [7]:
# 临时实验数据品种文件夹路径 TED_variety_path
TED_variety_path = os.path.join(str(temporary_experimental_data_dir), str(variety))

# 时间范围文件夹路径 dataset_range_boundary_path
dataset_range_boundary_path = os.path.join(TED_variety_path, dataset_range_boundary)

# 采样刻度文件路径 tick_granularity_resample_path
tick_granularity_resample_path = os.path.join(dataset_range_boundary_path, f'tick_granularity_{tick_granularity}')

# 驱动轴数据文件路径 resample_axis_data_path
resample_axis_data_path = os.path.join(tick_granularity_resample_path, resample_axis)



tick_type_dict = {
    'time_axis_data': 'Nrow',
    'volume_axis_data': 'volume',
    'money_axis_data': 'money'
}
if resample_axis in tick_type_dict.keys():
    tick_type = tick_type_dict[resample_axis]
else: raise ValueError

# 主力片段拼接重采样序列文件路径 main_continuous_resampled_timeseries_path
main_continuous_resampled_timeseries_path = os.path.join(resample_axis_data_path, f'main_continuous_{tick_type}_timeseries.csv')
main_continuous_resampled_timeseries = pd.read_csv(main_continuous_resampled_timeseries_path, dtype = {'date': str})

# 主力片段拼接重采样时间索引路径 main_continuous_resampled_date_index_path
main_continuous_resampled_date_index_path = os.path.join(resample_axis_data_path, f'main_continuous_{tick_type}_timeseries.csv')
main_continuous_resampled_date_index = pd.read_csv(main_continuous_resampled_date_index_path, dtype = {'date': str})

# 同步压缩变换 SWT 数据路径 CWT_SWT_path
CWT_SWT_path = os.path.join(resample_axis_data_path, f'CWT_SWT')
os.makedirs(CWT_SWT_path, exist_ok = True)

# SWT 实部数据路径 SWT_real_path
SWT_real_path = os.path.join(CWT_SWT_path, f'SWT_real.csv')
SWT_real = pd.read_csv(SWT_real_path)

# SWT 虚部数据路径 SWT_imag_path
SWT_imag_path = os.path.join(CWT_SWT_path, f'SWT_imag.csv')
SWT_imag = pd.read_csv(SWT_imag_path)

# SWT 幅值数据路径 SWT_magnitude_path
SWT_magnitude_path = os.path.join(CWT_SWT_path, f'SWT_magnitude.csv')
SWT_magnitude = pd.read_csv(SWT_magnitude_path)

# SWT 相位数据路径 SWT_phase_path
SWT_phase_path = os.path.join(CWT_SWT_path, f'SWT_phase.csv')
SWT_phase = pd.read_csv(SWT_phase_path)

# SWT 频率映射表路径 frequencies_map_path
frequencies_map_path = os.path.join(CWT_SWT_path, f'frequencies_map.csv')
frequencies_map = pd.read_csv(frequencies_map_path)

# SWT 小波变换衍生特征路径 SWT_features_path
SWT_features_path = os.path.join(CWT_SWT_path, f'SWT_features.csv')

In [13]:
# SWT_real：swt_endpoint 的实部，反映该频率成分的余弦分量振幅
# SWT_imag：swt_endpoint 的虚部，反映该频率成分的正弦分量振幅
# SWT_magnitude：幅值（模），即 sqrt(real² + imag²)，代表该频率成分在窗口末端的瞬时强度
# SWT_phase：相位（弧度），即 np.arctan2(imag, real)，表示该频率成分在窗口末端的相位偏移

import gc; gc.collect()
freq_grid = pd.to_numeric(frequencies_map['freq_str'], errors = 'coerce').values  # 无法转换的变为 NaN

display(frequencies_map.head())
display(SWT_magnitude.tail().T.head().T)
display(SWT_magnitude.tail().T.tail().T)

,frequency,freq_str,real_col,imag_col,magnitude_col,phase_col
0,0.011719,0.011719,real_f0.011719Hz,imag_f0.011719Hz,magnitude_f0.011719Hz,phase_f0.011719Hz
1,0.016654,0.016654,real_f0.016654Hz,imag_f0.016654Hz,magnitude_f0.016654Hz,phase_f0.016654Hz
2,0.021589,0.021589,real_f0.021589Hz,imag_f0.021589Hz,magnitude_f0.021589Hz,phase_f0.021589Hz
3,0.026524,0.026524,real_f0.026524Hz,imag_f0.026524Hz,magnitude_f0.026524Hz,phase_f0.026524Hz
4,0.031459,0.031459,real_f0.031459Hz,imag_f0.031459Hz,magnitude_f0.031459Hz,phase_f0.031459Hz


,magnitude_f0.011719Hz,magnitude_f0.016654Hz,magnitude_f0.021589Hz,magnitude_f0.026524Hz,magnitude_f0.031459Hz
64145,0.006929,0.005061,0.002664,0.004865,0.008294
64146,0.013344,0.016918,0.011922,0.002300,0.006105
64147,0.004454,0.008968,0.004754,0.000000,0.000350
64148,0.006716,0.012119,0.007790,0.000625,0.003019
64149,0.006041,0.004632,0.002752,0.006132,0.000951


,magnitude_f0.475616Hz,magnitude_f0.480551Hz,magnitude_f0.485486Hz,magnitude_f0.490421Hz,magnitude_f0.495356Hz
64145,0.0,0.000000,0.000000,0.000000,0.000000
64146,0.0,0.000729,0.000000,0.000772,0.000787
64147,0.0,0.000927,0.000000,0.000000,0.000935
64148,0.0,0.000762,0.000000,0.000000,0.000754
64149,0.0,0.000000,0.000991,0.000000,0.000993


In [16]:
# 能量 (energy) 幅值的平方，代表该频率成分的能量
energy = SWT_magnitude ** 2

# 总能量 (total_energy) 窗口内所有频率能量之和，反映信号全局强度
total_energy = energy.sum(axis = 1)

# 相对能量 (relative_energy) 每个频率能量占总能量的比例，反映能量在频域的分布
total_energy = energy.sum(axis = 1)   # 每行总能量
relative_energy = energy.div(total_energy, axis = 0)  # 归一化

# 频谱质心 (centroid) 幅值加权平均频率，表示频谱的重心位置
centroid = (SWT_magnitude * freq_grid).sum(axis = 1) / SWT_magnitude.sum(axis = 1)
centroid = centroid.fillna(0)   # 处理除零

# 频谱扩散 (spread) 幅值加权的频率标准差，反映频谱的离散程度
diff = (freq_grid - centroid.values.reshape(-1, 1)) ** 2
weighted_var = (diff * SWT_magnitude).sum(axis = 1) / SWT_magnitude.sum(axis = 1)
spread = np.sqrt(weighted_var.fillna(0))

# 主频率 (dominant_freq) 能量最大的频率点，代表信号中最主要的振荡频率
dominant_freq = pd.Series(index = energy.index, dtype = 'object') # 创建空序列存放结果
valid_mask = energy.notna().any(axis = 1) # 仅对至少有一个非 NaN 值的行计算
dominant_freq[valid_mask] = energy.loc[valid_mask].idxmax(axis = 1)  # 返回列名
dominant_freq[valid_mask] = dominant_freq[valid_mask].apply(lambda col: float(col.split('_')[1].replace('Hz', '').replace('f', ''))) # 将列名提取为频率值

# 能量集中度 - 峰值能量占比 (peak_energy_ratio) 最大相对能量，值越大表示能量越集中于单一频率
peak_energy_ratio = relative_energy.max(axis = 1)

# 频谱熵 (spectral_entropy) 基于相对能量的香农熵，越低表示能量分布越集中
relative_energy_safe = relative_energy.clip(lower = 1e-12) # 避免 log(0)
spectral_entropy = - (relative_energy_safe * np.log(relative_energy_safe)).sum(axis = 1)

# 高频 / 低频能量比 (high_low_energy_ratio) 以频率网格中点为界，高频能量与低频能量的比值
middle_point = len(freq_grid) // 2
low_freq_energy = energy.iloc[:, :middle_point].sum(axis = 1)
high_freq_energy = energy.iloc[:, middle_point:].sum(axis = 1)
high_low_energy_ratio = high_freq_energy / (low_freq_energy + 1e-12)

# 频谱偏度 (skewness) 衡量频谱分布的不对称性（正偏 → 高频有长尾，负偏 → 低频有长尾）
diff = (freq_grid - centroid.values.reshape(-1, 1))
diff3 = diff ** 3
skewness = (diff3 * SWT_magnitude).sum(axis = 1) / (SWT_magnitude.sum(axis = 1) * (spread ** 3))
skewness = skewness.fillna(0)

# 频谱峰度 (kurtosis) 衡量频谱的尖锐程度（>0 表示比正态分布更尖锐，有尖峰）
diff = (freq_grid - centroid.values.reshape(-1, 1))
diff4 = diff ** 4
num_kurt = (diff4 * SWT_magnitude).sum(axis = 1)
den_kurt = (SWT_magnitude.sum(axis = 1) * (spread ** 4))
kurtosis = num_kurt / den_kurt - 3        # 减去 3 使正态分布峰度为 0
kurtosis = kurtosis.fillna(0)

# 相位一致性 (phase_coherence) 基于相位方差的指标，值越大表示各频率相位越稳定，可暗示谐波关系
phase_variance = SWT_phase.var(axis = 1)
phase_coherence = 1.0 / (1.0 + phase_variance)

# 有效频率数量 (effective_freq_count) 能量超过最大能量 1% 的频率个数，反映主要频率成分数量
threshold = 0.01 * energy.max(axis = 1)
effective_freq_count = (energy > threshold.values.reshape(-1, 1)).sum(axis = 1)

# 频谱带宽 (spectral_bandwidth) 覆盖 90% 累积能量（5% ~ 95%）的频率区间宽度
energy_row_cumsum = relative_energy.cumsum(axis = 1)  # 对每行累积相对能量，用于查找 5% 和 95% 的分位点对应的频率索引
row_lower_idx = (energy_row_cumsum < 0.05).sum(axis = 1)          # 第一个累积 ≥ 0.05 的位置
row_upper_idx = (energy_row_cumsum < 0.95).sum(axis = 1)          # 第一个累积 ≥ 0.95 的位置
row_upper_idx = row_upper_idx.clip(upper = len(freq_grid) - 1)  # 处理边界：若没有达到 95%，取最后一个频率
spectral_bandwidth = freq_grid[row_upper_idx] - freq_grid[row_lower_idx]

# 中位数频率 (median_freq) 累积能量达到 50% 时的频率
energy_row_cumsum = relative_energy.cumsum(axis = 1)  # 对每行累积相对能量
row_median_idx = (energy_row_cumsum < 0.5).sum(axis = 1)
row_median_idx = row_median_idx.clip(upper = len(freq_grid) - 1)
median_freq = freq_grid[row_median_idx]

#  频率四分位距 (iqr_freq) 25%~75% 累积能量对应的频率宽度，描述中间 50% 能量的覆盖范围
energy_row_cumsum = relative_energy.cumsum(axis = 1)  # 对每行累积相对能量
row_q25_idx = (energy_row_cumsum < 0.25).sum(axis = 1)
row_q75_idx = (energy_row_cumsum < 0.75).sum(axis = 1)
row_q25_idx = row_q25_idx.clip(upper = len(freq_grid) - 1)
row_q75_idx = row_q75_idx.clip(upper = len(freq_grid) - 1)
iqr_freq = freq_grid[row_q75_idx] - freq_grid[row_q25_idx]

In [17]:
SWT_features = pd.DataFrame({
    'centroid': centroid,   # 频谱质心：幅值加权平均频率，∑(f * |W|) / ∑|W|
    'spread': spread,   # 频谱扩散：幅值加权的频率标准差，sqrt(∑((f - centroid)² * |W|) / ∑|W|)
    'dominant_freq': dominant_freq, # 主频率：能量最大的频率点，argmax(energy) 对应的频率值
    'peak_energy_ratio': peak_energy_ratio, # 峰值能量占比：最大相对能量，max(relative_energy)
    'spectral_entropy': spectral_entropy,   # 频谱熵：基于相对能量的香农熵，-∑(p * log(p))，其中 p = relative_energy
    'high_low_energy_ratio': high_low_energy_ratio, # 高频/低频能量比：高频段能量和 / 低频段能量和（以频率中点为界）
    'skewness': skewness,   # 频谱偏度：衡量频谱分布不对称性，∑((f - centroid)³ * |W|) / (∑|W| * spread³)
    'kurtosis': kurtosis,   # 频谱峰度：衡量频谱尖锐程度，∑((f - centroid)⁴ * |W|) / (∑|W| * spread⁴) - 3
    'phase_coherence': phase_coherence, # 相位一致性：1 / (1 + var(phase))，值越大相位越稳定
    'effective_freq_count': effective_freq_count,   # 有效频率数量：能量超过最大能量 1% 的频率个数
    'spectral_bandwidth': spectral_bandwidth,   # 频谱带宽：覆盖 90% 累积能量的频率区间宽度（5%~95%）
    'median_freq': median_freq, # 中位数频率：累积能量达到 50% 时的频率
    'iqr_freq': iqr_freq,   # 频率四分位距：25%~75 %累积能量对应的频率宽度
    'total_energy': total_energy    # 总能量：所有频率能量之和，∑energy
}, index = SWT_magnitude.index)

# SWT 小波变换衍生特征路径 SWT_features_path
SWT_features_path = os.path.join(CWT_SWT_path, f'SWT_features.csv')
SWT_features.to_csv(SWT_features_path, index = False)
import gc; gc.collect()

3878